In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from pytorch3d.vis.plotly_vis import plot_scene
from data_tools import adv_dataset, adversarial_patch_3d

from attack_utils import *

CKPT_PATH = "output/train/query_attack_pointrcnn/final_adversarial_patch_checkpoint.pt"

In [6]:
state_dict = torch.load(CKPT_PATH)

In [7]:
universal_adv_patch_car = single_sphere(scale=adv_dataset.CAR_ADV_PATCH_SCALE)
universal_adv_patch_car.load_parameter(state_dict["universal_adv_patch_car"])

print(state_dict["universal_adv_patch_car"])

mesh vertex count : torch.Size([162, 3])
[tensor([-0.0224,  0.0171,  0.0000], device='cuda:0', requires_grad=True), tensor([0.0660], device='cuda:0', requires_grad=True), tensor([[ 1.5025e-02,  9.6227e-03,  0.0000e+00],
        [ 3.1811e-02,  3.7575e-02,  0.0000e+00],
        [ 5.0026e-03,  1.4803e-02,  0.0000e+00],
        [-2.3306e-02, -4.0711e-02,  0.0000e+00],
        [ 0.0000e+00, -2.9432e-02,  0.0000e+00],
        [ 0.0000e+00,  6.1798e-02,  0.0000e+00],
        [ 0.0000e+00, -5.4356e-02,  0.0000e+00],
        [ 0.0000e+00, -5.0342e-03,  0.0000e+00],
        [-9.4728e-03,  0.0000e+00,  0.0000e+00],
        [-1.1629e-01,  0.0000e+00,  0.0000e+00],
        [-1.4227e-02,  0.0000e+00,  0.0000e+00],
        [-5.0536e-02,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [-3.6259e-02, -5.7220e-02,  0.0000e+00],
        [-2.1071e-02, -3.8929e-02,  0.0000e+00],
        [-2.0620e-02, -2.9540e-02,  0.0000e+00],
        [-8.0418e-02, -6.6613e-02,  0.0000e+0

In [8]:
fig = plot_scene({
                    "original": {
                        "mesh_1": universal_adv_patch_car.get_basic_meshes()
                    },
                    "adversarial": {
                        "mesh_1": universal_adv_patch_car.get_deformed_meshes()
                    },
                }, ncols=2)
fig.update_layout(height=400, width=800)
fig.show()

In [3]:
from data_tools import simple_cubic_meshes, join_meshes_as_batch
from pytorch3d.vis.plotly_vis import plot_scene

test = simple_cubic_meshes(cubic_level=2)
meshes_batch = join_meshes_as_batch(test.get_deformed_lattice())

In [4]:
fig = plot_scene({
                    "original": {
                        "mesh_1": meshes_batch
                    }
                }, ncols=1)
fig.update_layout(height=400, width=400)
fig.show()

In [6]:
import pandas as pd

def parse_table(data):
    # 将数据转换为DataFrame
    lines = data.strip().split('\n')
    rows = [line.split() for line in lines]
    columns = ["Category", "Value1", "Value2", "Value3", "Value4"]

    df = pd.DataFrame(rows, columns=columns)

    # 将数值列转换为浮点数
    df[["Value1", "Value2", "Value3", "Value4"]] = df[["Value1", "Value2", "Value3", "Value4"]].astype(float)

    print(df)

    clean_values = df.loc[df['Category'] == 'Clean', ["Value1", "Value2", "Value3", "Value4"]].values[0]

    # 计算每个类别相对于Clean的百分比
    percentage_df = df.copy()
    percentage_df[["Value1", "Value2", "Value3", "Value4"]] = 100 - df[["Value1", "Value2", "Value3", "Value4"]].div(clean_values) * 100

    print("Original DataFrame:")
    print(df)
    print("\nPercentage DataFrame:")
    print(percentage_df)

In [9]:
data_1 = """
Vanilla@BEV	76.9694	83.1086	88.1049	87.5583
Vanilla@3D	66.5986	66.2274	76.1004	73.9477
Car@BEV	76.8111	81.9277	87.8725	87.3614
Car@3D	65.8241	64.8201	75.0164	72.1968
"""

data_2 = """
Vanilla@BEV	76.9694	83.1086	88.1049	87.5583
Vanilla@3D	66.5986	66.2274	76.1004	73.9477
Car@BEV	76.8111	81.9277	87.8725	87.3614
Car@3D	65.8241	64.8201	75.0164	72.1968
"""


parse_table(data_1)

parse_table(data_2)

      Category   Value1   Value2   Value3   Value4
0        Clean  86.7564  85.6744  88.9160  89.1513
1      Vanilla  80.6136  76.9694  83.1086  88.1049
2   FullAttack  79.4522  74.2181  86.4966  77.4980
3    IouFrozen  78.6085  72.7279  86.9074  76.9161
4  LogitFrozen  76.5190  74.8475  85.5587  76.4687
Original DataFrame:
      Category   Value1   Value2   Value3   Value4
0        Clean  86.7564  85.6744  88.9160  89.1513
1      Vanilla  80.6136  76.9694  83.1086  88.1049
2   FullAttack  79.4522  74.2181  86.4966  77.4980
3    IouFrozen  78.6085  72.7279  86.9074  76.9161
4  LogitFrozen  76.5190  74.8475  85.5587  76.4687

Percentage DataFrame:
      Category     Value1     Value2    Value3     Value4
0        Clean   0.000000   0.000000  0.000000   0.000000
1      Vanilla   7.080515  10.160561  6.531333   1.173735
2   FullAttack   8.419206  13.371906  2.720995  13.071374
3    IouFrozen   9.391699  15.111282  2.258986  13.724085
4  LogitFrozen  11.800167  12.637264  3.775811  14.2259

In [12]:
import pandas as pd
import numpy as np
def parse_table(data):
    # 将数据转换为DataFrame
    lines = data.strip().split('\n')
    rows = [line.split() for line in lines[1:]]
    columns = lines[0].split()
    
    df = pd.DataFrame(rows, columns=columns)

    # 将数值列转换为浮点数
    df[columns[1:]] = df[columns[1:]].astype(float)

    print(df)
    
    return df

    # clean_values = df.loc[df['Category'] == 'Clean', ["Value1", "Value2", "Value3", "Value4"]].values[0]

    # # 计算每个类别相对于Clean的百分比
    # percentage_df = df.copy()
    # percentage_df[["Value1", "Value2", "Value3", "Value4"]] = 100 - df[["Value1", "Value2", "Value3", "Value4"]].div(clean_values) * 100

    # print("Original DataFrame:")
    # print(df)
    # print("\nPercentage DataFrame:")
    # print(percentage_df)


In [19]:
table_1 = """
Metric	PointRCNN	PVRCNN	VoxelRCNN(Car)	Second
Clean@BEV	85.6744	88.9160	89.1513	88.7158
Clean@3D	78.6668	66.2274	76.1004	73.9477
Vanilla@BEV	76.9694	83.1086	88.1049	87.5583
Vanilla@3D	66.5986	66.2274	76.1004	73.9477
Car@BEV	76.8111	81.9277	87.8725	87.3614
Car@3D	65.8241	64.8201	75.0164	72.1968
"""
table_1:pd.DataFrame = parse_table(table_1)
clean_bev = table_1.loc[table_1['Metric'] == 'Clean@BEV'].values[0, 1:].astype(float)
clean_3d = table_1.loc[table_1['Metric'] == 'Clean@3D'].values[0, 1:].astype(float)
car_bev = table_1.loc[table_1['Metric'] == 'Car@BEV'].values[0, 1:].astype(float)
car_3d = table_1.loc[table_1['Metric'] == 'Car@3D'].values[0, 1:].astype(float)
# table_1 = table_1.append(new_row, ignore_index=True)
print(np.round((1 - (car_bev/clean_bev))*100, 2))
print(np.round((1 - (car_3d/clean_3d))*100, 2))

        Metric  PointRCNN   PVRCNN  VoxelRCNN(Car)   Second
0    Clean@BEV    85.6744  88.9160         89.1513  88.7158
1     Clean@3D    78.6668  66.2274         76.1004  73.9477
2  Vanilla@BEV    76.9694  83.1086         88.1049  87.5583
3   Vanilla@3D    66.5986  66.2274         76.1004  73.9477
4      Car@BEV    76.8111  81.9277         87.8725  87.3614
5       Car@3D    65.8241  64.8201         75.0164  72.1968
[10.35  7.86  1.43  1.53]
[16.33  2.12  1.42  2.37]


In [22]:
table_1 = """
Metric	PointPIllar	PVRCNN	VoxelRCNN(Car)	Second
Clean@BEV	86.7564	88.9160	89.1513	88.7158
Clean@3D	77.5743	66.2274	76.1004	73.9477
Vanilla@BEV	80.6136	83.1086	88.1049	87.5583
Vanilla@3D	57.1795	66.2274	76.1004	73.9477
Car@BEV	79.3103	82.9085	82.7351	86.9980
Car@3D	53.3262	66.0200	65.8373	69.7959
"""
table_1:pd.DataFrame = parse_table(table_1)
clean_bev = table_1.loc[table_1['Metric'] == 'Clean@BEV'].values[0, 1:].astype(float)
clean_3d = table_1.loc[table_1['Metric'] == 'Clean@3D'].values[0, 1:].astype(float)
car_bev = table_1.loc[table_1['Metric'] == 'Car@BEV'].values[0, 1:].astype(float)
car_3d = table_1.loc[table_1['Metric'] == 'Car@3D'].values[0, 1:].astype(float)
# table_1 = table_1.append(new_row, ignore_index=True)
print(np.round((1 - (car_bev/clean_bev))*100, 2))
print(np.round((1 - (car_3d/clean_3d))*100, 2))

        Metric  PointPIllar   PVRCNN  VoxelRCNN(Car)   Second
0    Clean@BEV      86.7564  88.9160         89.1513  88.7158
1     Clean@3D      77.5743  66.2274         76.1004  73.9477
2  Vanilla@BEV      80.6136  83.1086         88.1049  87.5583
3   Vanilla@3D      57.1795  66.2274         76.1004  73.9477
4      Car@BEV      79.3103  82.9085         82.7351  86.9980
5       Car@3D      53.3262  66.0200         65.8373  69.7959
[8.58 6.76 7.2  1.94]
[31.26  0.31 13.49  5.61]


In [23]:
table_1 = """
Metric	PointPIllar	PVRCNN	VoxelRCNN(Car)	Second
Clean@BEV	86.7564	88.9160	89.1513	88.7158
Clean@3D	77.5743	66.2274	76.1004	73.9477
Vanilla@BEV	80.6136	83.1086	88.1049	87.5583
Vanilla@3D	57.1795	66.2274	76.1004	73.9477
Car@BEV	80.0401	82.2935	87.8767	87.2270
Car@3D	56.2568	64.8246	74.1950	70.3205
"""
table_1:pd.DataFrame = parse_table(table_1)
clean_bev = table_1.loc[table_1['Metric'] == 'Clean@BEV'].values[0, 1:].astype(float)
clean_3d = table_1.loc[table_1['Metric'] == 'Clean@3D'].values[0, 1:].astype(float)
car_bev = table_1.loc[table_1['Metric'] == 'Car@BEV'].values[0, 1:].astype(float)
car_3d = table_1.loc[table_1['Metric'] == 'Car@3D'].values[0, 1:].astype(float)
# table_1 = table_1.append(new_row, ignore_index=True)
print(np.round((1 - (car_bev/clean_bev))*100, 2))
print(np.round((1 - (car_3d/clean_3d))*100, 2))

        Metric  PointPIllar   PVRCNN  VoxelRCNN(Car)   Second
0    Clean@BEV      86.7564  88.9160         89.1513  88.7158
1     Clean@3D      77.5743  66.2274         76.1004  73.9477
2  Vanilla@BEV      80.6136  83.1086         88.1049  87.5583
3   Vanilla@3D      57.1795  66.2274         76.1004  73.9477
4      Car@BEV      80.0401  82.2935         87.8767  87.2270
5       Car@3D      56.2568  64.8246         74.1950  70.3205
[7.74 7.45 1.43 1.68]
[27.48  2.12  2.5   4.91]
